# Modélisation

## Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Chargement des données préparées

In [2]:
hotel_booking = pd.read_csv("../data/processed/hotel_booking_prepared.csv")
flight = pd.read_csv("../data/processed/flight_prepared.csv")

## Modélisation du dataset hôtel

### Définition du problème : prédiction de l'annulation
L'objectif est de construire un modèle capable de prédire si une réservation d'hôtel sera annulée (`is_canceled`).

Il s'agit d'un problème de **classification binaire** :

- 0 : la réservation est maintenue
- 1 : la réservation est annulée

### Définition de X et Y

La variable cible est `is_canceled`.

Toutes les autres variables serviront à entraîner le modèle.

In [3]:
X_hotel = hotel_booking.drop(columns="is_canceled")
y_hotel = hotel_booking["is_canceled"]

In [4]:
print(X_hotel.shape)
print(y_hotel.shape)

(87387, 30)
(87387,)


### Séparation Train/Test


Le jeu de données est séparé en deux parties :

- 80 % pour entraîner les modèles
- 20 % pour évaluer leurs performances sur des données jamais vues.

In [5]:
from sklearn.model_selection import train_test_split

X_train_hotel, X_test_hotel, y_train_hotel, y_test_hotel = train_test_split(
    X_hotel,
    y_hotel,
    test_size=0.2,
    random_state=42
)

In [6]:
print(X_train_hotel.shape)
print(X_test_hotel.shape)

(69909, 30)
(17478, 30)


### Standardisation des variables

Certaines méthodes de machine learning que nous allons utiliser pour la modléisation, comme la régression logistique, sont sensibles à l'échelle des variables.

Une standardisation est donc appliquée afin que toutes les variables soient sur une même échelle.

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_hotel_scaled = scaler.fit_transform(X_train_hotel)
X_test_hotel_scaled = scaler.transform(X_test_hotel)

In [8]:
X_train_hotel_scaled.shape

(69909, 30)

### Test des différents modèles

#### Baseline : Dummy Classifier

In [9]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_hotel, y_train_hotel)

y_hotel_pred_baseline = baseline.predict(X_test_hotel)

In [10]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test_hotel, y_hotel_pred_baseline)

0.7259411832017393

Le DummyClassifier sert de modèle de référence. Il prédit systématiquement la classe majoritaire sans exploiter les caractéristiques des réservations.

Il obtient une accuracy de 72,6 %. Cette performance constitue le seuil minimal à dépasser. Les modèles de machine learning suivants devront obtenir de meilleurs résultats pour démontrer leur capacité à apprendre les relations présentes dans les données.

#### Régression logistique

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

log_reg = LogisticRegression(
    random_state=42,
    max_iter=1000
)

log_reg.fit(X_train_hotel_scaled, y_train_hotel)

y_hotel_pred_log_reg = log_reg.predict(X_test_hotel_scaled)

In [12]:
# Evalutation du modèle 
accuracy_log_reg = accuracy_score(y_test_hotel, y_hotel_pred_log_reg)

print("Accuracy - Régression logistique :", accuracy_log_reg)

Accuracy - Régression logistique : 0.7728000915436549


In [13]:
print(classification_report(y_test_hotel, y_hotel_pred_log_reg))

              precision    recall  f1-score   support

           0       0.79      0.93      0.86     12688
           1       0.65      0.37      0.47      4790

    accuracy                           0.77     17478
   macro avg       0.72      0.65      0.66     17478
weighted avg       0.76      0.77      0.75     17478



In [14]:
confusion_matrix(y_test_hotel, y_hotel_pred_log_reg)

array([[11752,   936],
       [ 3035,  1755]])

La régression logistique améliore les performances par rapport au modèle de référence en atteignant une accuracy de **77,3 %**, contre **72,6 %** pour le DummyClassifier.

Le modèle identifie correctement la majorité des réservations non annulées. En revanche, il rencontre davantage de difficultés pour détecter les réservations annulées, comme en témoigne le rappel de **37 %** sur cette classe.

Ces résultats montrent que des modèles plus complexes, tels que Random Forest ou XGBoost, pourraient mieux capturer les relations non linéaires présentes dans les données et améliorer la détection des annulations.

### Random Forest

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42
)

rf.fit(X_train_hotel, y_train_hotel)

y_hotel_pred_rf = rf.predict(X_test_hotel)

In [16]:
accuracy_rf = accuracy_score(y_test_hotel, y_hotel_pred_rf)

print("Accuracy - Random Forest :", accuracy_rf)

Accuracy - Random Forest : 0.8494679025060076


In [17]:
print(classification_report(y_test_hotel, y_hotel_pred_rf))

              precision    recall  f1-score   support

           0       0.87      0.93      0.90     12688
           1       0.78      0.63      0.70      4790

    accuracy                           0.85     17478
   macro avg       0.82      0.78      0.80     17478
weighted avg       0.84      0.85      0.84     17478



In [18]:
confusion_matrix(y_test_hotel, y_hotel_pred_rf)

array([[11829,   859],
       [ 1772,  3018]])

Le modèle Random Forest améliore sensiblement les performances par rapport à la régression logistique.

L'accuracy passe d'environ 77 % à près de 85 %, tandis que le rappel de la classe représentant les annulations augmente fortement.

Ce modèle détecte davantage de réservations annulées tout en conservant une bonne précision. Il constitue donc un candidat plus performant pour ce problème de classification.

#### XGboost

In [19]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train_hotel, y_train_hotel)

y_hotel_pred_xgb = xgb.predict(X_test_hotel)

In [20]:
accuracy_xgb = accuracy_score(y_test_hotel, y_hotel_pred_xgb)

print("Accuracy - XGBoost :", accuracy_xgb)

Accuracy - XGBoost : 0.8530152191326239


In [21]:
print(classification_report(y_test_hotel, y_hotel_pred_xgb))

              precision    recall  f1-score   support

           0       0.88      0.92      0.90     12688
           1       0.76      0.68      0.72      4790

    accuracy                           0.85     17478
   macro avg       0.82      0.80      0.81     17478
weighted avg       0.85      0.85      0.85     17478



In [22]:
confusion_matrix(y_test_hotel, y_hotel_pred_xgb)

array([[11671,  1017],
       [ 1552,  3238]])

XGBoost est comparé aux modèles précédents afin de déterminer lequel offre le meilleur compromis entre précision, rappel et score F1.

Le modèle présentant les meilleures performances globales sera retenu pour la suite du projet.

### Conclusion pour les tests réalisés

In [23]:
import pandas as pd
from sklearn.metrics import recall_score, f1_score

resultats = pd.DataFrame({
    "Modèle": [
        "Baseline",
        "Régression logistique",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test_hotel, y_hotel_pred_baseline),
        accuracy_log_reg,
        accuracy_rf,
        accuracy_xgb
    ],
    "Recall (classe 1)": [
        recall_score(y_test_hotel, y_hotel_pred_baseline),
        recall_score(y_test_hotel, y_hotel_pred_log_reg),
        recall_score(y_test_hotel, y_hotel_pred_rf),
        recall_score(y_test_hotel, y_hotel_pred_xgb)
    ],
    "F1-score (classe 1)": [
        f1_score(y_test_hotel, y_hotel_pred_baseline),
        f1_score(y_test_hotel, y_hotel_pred_log_reg),
        f1_score(y_test_hotel, y_hotel_pred_rf),
        f1_score(y_test_hotel, y_hotel_pred_xgb)
    ]
})

resultats.round(3)

,Modèle,Accuracy,Recall (classe 1),F1-score (classe 1)
0,Baseline,0.726,0.000,0.000
1,Régression logistique,0.773,0.366,0.469
2,Random Forest,0.849,0.630,0.696
3,XGBoost,0.853,0.676,0.716


Quatre modèles ont été évalués afin de prédire les annulations de réservation :

- Baseline (Dummy Classifier)
- Régression logistique
- Random Forest
- XGBoost

Le modèle XGBoost obtient les meilleures performances globales sur les données de test.

Il présente la meilleure accuracy tout en améliorant le rappel et le score F1 sur la classe correspondant aux réservations annulées.

Il est donc retenu comme modèle final pour ce jeu de données.

### Validation croisée

Les performances observées sur un unique découpage des données peuvent varier selon la répartition des observations.

Une validation croisée (*Cross-Validation*) est donc réalisée afin d'évaluer la robustesse du modèle XGBoost sur plusieurs sous-ensembles du jeu de données.

Une validation croisée à **5 plis (5-Fold Cross Validation)** est utilisée.

In [24]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier

X_hotel_hotel = hotel_booking.drop(columns="is_canceled")
y_hotel_hotel = hotel_booking["is_canceled"]

xgb_hotel = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores_hotel = cross_val_score(
    estimator=xgb_hotel,
    X=X_hotel,
    y=y_hotel,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("Scores :", cv_scores_hotel)
print("Accuracy moyenne :", cv_scores_hotel.mean())
print("Écart-type :", cv_scores_hotel.std())

Scores : [0.85490331 0.8479231  0.85335012 0.85506666 0.85460891]
Accuracy moyenne : 0.8531704202504967
Écart-type : 0.002692007053686881


Une validation croisée stratifiée à 5 plis a été réalisée afin d'évaluer la robustesse du modèle XGBoost.

Le modèle obtient une accuracy moyenne de **85,3 %** avec un écart-type très faible (**0,003**), ce qui montre que ses performances sont stables quel que soit le découpage des données.

Ces résultats confirment que le modèle généralise correctement et qu'il ne dépend pas d'un découpage particulier du jeu de données.

### Grid Search
Après validation croisée, une optimisation des hyperparamètres est réalisée sur le modèle XGBoost.

L'objectif est de tester plusieurs combinaisons de paramètres afin d'améliorer les performances du modèle.

In [25]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

xgb_grid = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1]
}

grid_search_xgb = GridSearchCV(
    estimator=xgb_grid,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_search_xgb.fit(X_hotel, y_hotel)

print("Meilleurs paramètres :", grid_search_xgb.best_params_)
print("Meilleure accuracy moyenne :", grid_search_xgb.best_score_)

Meilleurs paramètres : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Meilleure accuracy moyenne : 0.8491423383687909


In [26]:
best_xgb = grid_search_xgb.best_estimator_

In [27]:
y_hotel_pred_best_xgb = best_xgb.predict(X_test_hotel)

accuracy_best_xgb = accuracy_score(y_test_hotel, y_hotel_pred_best_xgb)

print("Accuracy - XGBoost optimisé :", accuracy_best_xgb)
print(classification_report(y_test_hotel, y_hotel_pred_best_xgb))
confusion_matrix(y_test_hotel, y_hotel_pred_best_xgb)

Accuracy - XGBoost optimisé : 0.8546172330930313
              precision    recall  f1-score   support

           0       0.88      0.92      0.90     12688
           1       0.77      0.67      0.72      4790

    accuracy                           0.85     17478
   macro avg       0.83      0.80      0.81     17478
weighted avg       0.85      0.85      0.85     17478



array([[11735,   953],
       [ 1588,  3202]])

Le GridSearchCV a permis d'identifier la meilleure combinaison d'hyperparamètres pour le modèle XGBoost.

Le modèle optimisé obtient une accuracy de **85,5 %**, très légèrement supérieure à celle du modèle initial.

L'amélioration reste limitée, ce qui indique que le modèle XGBoost initial était déjà bien adapté au problème. Le modèle optimisé est néanmoins retenu comme modèle final pour la prédiction des annulations de réservation.

## Modélisation du dataset vols

### Définition du problème : prédiction du prix des billets

L'objectif est de construire un modèle capable de prédire le prix d'un billet d'avion (`price`) à partir des caractéristiques du vol.

Il s'agit d'un problème de **régression**, la variable cible étant une valeur numérique continue.

### Définition de X et Y

In [28]:
X_flight = flight.drop(columns="price")
y_flight = flight["price"]

In [29]:
print(X_flight.shape)
print(y_flight.shape)

(300153, 10)
(300153,)


### Séparation Train/Test

Le jeu de données est séparé en deux ensembles :

- **80 %** pour entraîner les modèles.
- **20 %** pour évaluer leurs performances sur des données jamais vues.

In [30]:
from sklearn.model_selection import train_test_split

X_train_flight, X_test_flight, y_train_flight, y_test_flight = train_test_split(
    X_flight,
    y_flight,
    test_size=0.2,
    random_state=42
)

In [31]:
print(X_train_flight.shape)
print(X_test_flight.shape)

(240122, 10)
(60031, 10)


### Standardisation 
Les variables explicatives sont standardisées afin de faciliter l'entraînement des modèles sensibles à l'échelle des données, notamment la régression linéaire.

In [32]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_flight_scaled = scaler.fit_transform(X_train_flight)
X_test_flight_scaled = scaler.transform(X_test_flight)

In [33]:
X_train_flight_scaled.shape

(240122, 10)

### Tests des différents modèles
#### Baseline : Dummy Classifier

In [34]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train_flight, y_train_flight)

y_flight_pred_baseline = baseline.predict(X_test_flight)

In [35]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae_baseline = mean_absolute_error(y_test_flight, y_flight_pred_baseline)
rmse_baseline = mean_squared_error(y_test_flight, y_flight_pred_baseline) ** 0.5
r2_baseline = r2_score(y_test_flight, y_flight_pred_baseline)

print("MAE :", mae_baseline)
print("RMSE :", rmse_baseline)
print("R² :", r2_baseline)

MAE : 19768.69424587037
RMSE : 22704.23535747973
R² : -5.741993791552602e-08


Le DummyRegressor sert de modèle de référence.

Il prédit simplement la moyenne des prix du jeu d'entraînement, sans utiliser les variables explicatives.

Les performances obtenues sont faibles (R² ≈ 0), ce qui est attendu. Les modèles suivants devront obtenir un MAE plus faible, un RMSE plus faible et un R² plus élevé pour démontrer qu'ils apprennent réellement les relations présentes dans les données.

#### Régression linéaire

In [36]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()

lin_reg.fit(X_train_flight_scaled, y_train_flight)

y_flight_pred_lin = lin_reg.predict(X_test_flight_scaled)

In [37]:
# Evaluation du modèle 
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae_lin = mean_absolute_error(y_test_flight, y_flight_pred_lin)
rmse_lin = mean_squared_error(y_test_flight, y_flight_pred_lin) ** 0.5
r2_lin = r2_score(y_test_flight, y_flight_pred_lin)

print("MAE :", mae_lin)
print("RMSE :", rmse_lin)
print("R² :", r2_lin)

MAE : 4622.18710336136
RMSE : 7013.558484851851
R² : 0.9045747930770209


La régression linéaire améliore très fortement les performances par rapport au DummyRegressor.

Le coefficient de détermination (R² ≈ 0,90) montre que le modèle explique une grande partie de la variabilité du prix des billets.

Les erreurs MAE et RMSE sont également nettement plus faibles que celles du modèle de référence.

#### Random Forest Regressor

In [38]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train_flight, y_train_flight)

y_flight_pred_rf = rf.predict(X_test_flight)

In [39]:
mae_rf = mean_absolute_error(y_test_flight, y_flight_pred_rf)
rmse_rf = mean_squared_error(y_test_flight, y_flight_pred_rf) ** 0.5
r2_rf = r2_score(y_test_flight, y_flight_pred_rf)

print("MAE :", mae_rf)
print("RMSE :", rmse_rf)
print("R² :", r2_rf)

MAE : 862.9694714258084
RMSE : 2319.055808038939
R² : 0.9895670130350649


Le Random Forest Regressor améliore fortement les performances par rapport à la régression linéaire.

Les erreurs MAE et RMSE diminuent de manière importante tandis que le coefficient de détermination atteint près de 0,99.

Ces résultats montrent que les relations entre les variables explicatives et le prix des billets sont en grande partie non linéaires, ce que le Random Forest est capable de modéliser efficacement.

#### XG Boost

In [40]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    objective="reg:squarederror"
)

xgb.fit(X_train_flight, y_train_flight)

y_flight_pred_xgb = xgb.predict(X_test_flight)

In [41]:
mae_xgb = mean_absolute_error(y_test_flight, y_flight_pred_xgb)
rmse_xgb = mean_squared_error(y_test_flight, y_flight_pred_xgb) ** 0.5
r2_xgb = r2_score(y_test_flight, y_flight_pred_xgb)

print("MAE :", mae_xgb)
print("RMSE :", rmse_xgb)
print("R² :", r2_xgb)

MAE : 2016.291748046875
RMSE : 3595.0970779660456
R² : 0.9749269485473633



Le XGBoost Regressor obtient d'excellentes performances avec un R² supérieur à 0,97.

Cependant, dans cette étude, le Random Forest Regressor reste le modèle le plus performant avec les erreurs les plus faibles (MAE et RMSE) ainsi que le meilleur coefficient de détermination.

Le Random Forest est donc retenu comme modèle final pour la prédiction du prix des billets d'avion.

### Conclusion des tests réalisés

In [42]:
import pandas as pd

resultats = pd.DataFrame({
    "Modèle": [
        "Dummy Regressor",
        "Régression linéaire",
        "Random Forest Regressor",
        "XGBoost Regressor"
    ],
    "MAE": [
        mae_baseline,
        mae_lin,
        mae_rf,
        mae_xgb
    ],
    "RMSE": [
        rmse_baseline,
        rmse_lin,
        rmse_rf,
        rmse_xgb
    ],
    "R²": [
        r2_baseline,
        r2_lin,
        r2_rf,
        r2_xgb
    ]
})

# Arrondir les valeurs pour une meilleure lisibilité
resultats = resultats.round(3)

resultats

,Modèle,MAE,RMSE,R²
0,Dummy Regressor,19768.694,22704.235,-0.000
1,Régression linéaire,4622.187,7013.558,0.905
2,Random Forest Regressor,862.969,2319.056,0.990
3,XGBoost Regressor,2016.292,3595.097,0.975


Quatre modèles ont été comparés afin de prédire le prix des billets d'avion.

Le **Dummy Regressor** sert de modèle de référence et obtient des performances très faibles, ce qui confirme que les variables du jeu de données apportent une réelle information.

La **régression linéaire** améliore fortement les résultats en expliquant plus de 90 % de la variance du prix des billets.

Le **Random Forest Regressor** est le modèle le plus performant. Il obtient les erreurs les plus faibles (MAE et RMSE) ainsi que le meilleur coefficient de détermination (R² ≈ 0,99). Il est capable de capturer les relations non linéaires entre les variables explicatives et le prix des billets.

Le **XGBoost Regressor** fournit également d'excellents résultats, mais reste légèrement moins performant que le Random Forest avec les paramètres utilisés.

Au regard des performances obtenues, **le Random Forest Regressor est retenu comme modèle final** pour la prédiction du prix des billets d'avion.

### Validation croisée

In [43]:
# On fait la validation croisée avec le modèle le plus performant soit Random Forest
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_flight = RandomForestRegressor(
    random_state=42
)

cv_scores_flight = cross_val_score(
    estimator=rf_flight,
    X=X_flight,
    y=y_flight,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

print("Scores :", cv_scores_flight)
print("R² moyen :", cv_scores_flight.mean())
print("Écart-type :", cv_scores_flight.std())

Scores : [0.98958202 0.98975748 0.98997839 0.98973574 0.98973025]
R² moyen : 0.9897567758914816
Écart-type : 0.00012712019938650954


La validation croisée à 5 plis confirme la robustesse du Random Forest Regressor.

Le modèle obtient un coefficient de détermination moyen (R²) d'environ **0,99** avec un écart-type très faible. Les performances restent donc très stables quelle que soit la partition du jeu de données, ce qui montre une bonne capacité de généralisation.

### Grid Search

In [44]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5]
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

grid_search_rf.fit(X_flight, y_flight)

print("Meilleurs paramètres :", grid_search_rf.best_params_)
print("Meilleur R² moyen :", grid_search_rf.best_score_)

/Users/emiliemoissette/Desktop/Projet 1 - AI Travel Planner/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Meilleurs paramètres : {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Meilleur R² moyen : 0.9900067855615486


In [45]:
best_rf = grid_search_rf.best_estimator_

best_rf.fit(X_train_flight, y_train_flight)

y_flight_pred_best_rf = best_rf.predict(X_test_flight)

mae_best = mean_absolute_error(y_test_flight, y_flight_pred_best_rf)
rmse_best = mean_squared_error(y_test_flight, y_flight_pred_best_rf) ** 0.5
r2_best = r2_score(y_test_flight, y_flight_pred_best_rf)

print("MAE :", mae_best)
print("RMSE :", rmse_best)
print("R² :", r2_best)

MAE : 879.730316758351
RMSE : 2284.9506965367978
R² : 0.9898716212888232


Le GridSearchCV a permis d'optimiser les hyperparamètres du modèle. Les performances restent excellentes et le R² est légèrement amélioré, tandis que la MAE augmente très légèrement. L'amélioration globale étant positive, le Random Forest optimisé est retenu comme modèle final.

## Conclusion

Ce projet avait pour objectif de développer des modèles de machine learning capables de répondre à deux problématiques différentes du domaine du voyage :
- Prédire l'annulation d'une réservation d'hôtel (classification).
- Prédire le prix d'un billet d'avion (régression).

Pour chacun des jeux de données, plusieurs modèles ont été entraînés et comparés à un modèle de référence (Dummy). Les performances ont ensuite été validées par une validation croisée et les meilleurs modèles ont été optimisés avec GridSearchCV.

Les résultats obtenus montrent que :
- Pour le dataset hôtel, XGBoost Classifier obtient les meilleures performances avec une accuracy d'environ 85 %, confirmée par une validation croisée stable.
- Pour le dataset vols, Random Forest Regressor fournit les meilleures prédictions avec un R² proche de 0,99, ainsi que des erreurs de prédiction très faibles. Les performances sont confirmées après validation croisée et optimisation des hyperparamètres.

Ce projet met en évidence l'importance de comparer plusieurs approches, d'évaluer leur robustesse grâce à la validation croisée et d'optimiser les hyperparamètres afin d'obtenir les meilleures performances possibles. Il illustre également l'application de techniques de machine learning à deux problématiques concrètes du secteur du voyage.

In [46]:
# Sauvegarde des modèles 
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(best_xgb, "../models/hotel_model.pkl")
joblib.dump(best_rf, "../models/flight_model.pkl")

print("Modèles sauvegardés avec succès.")

Modèles sauvegardés avec succès.
